In [50]:
from methods.arima import ArimaForecaster
from methods.naive_forecast import NaiveForecaster

import pandas as pd

import warnings

warnings.filterwarnings("ignore")

In [51]:
from utils.model_data_prep import prepare_data_for_modeling

In [52]:
train, test = prepare_data_for_modeling("MSFT", "1d",)

2025-04-08 19:52:11,302 - INFO - Successfully loaded data from data/MSFT/MSFT_1d.csv
2025-04-08 19:52:11,308 - INFO - Initial split: Train size=7875, Test size=1969
2025-04-08 19:52:11,310 - INFO - train_last_n is a float (1.0). Keeping last 7875 points (100.00%) of initial train set.
2025-04-08 19:52:11,311 - INFO - Final train set size after selecting last 7875 points: 7875


In [53]:
all_data = []
all_data.extend(train.values)
all_data.extend(test.values)

In [54]:
len(all_data) == len(train) + len(test)

True

In [55]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np


# Scale - split

In [56]:
scaler = MinMaxScaler(feature_range=(0,1))

scaler.fit(np.array(all_data).reshape(-1,1))

MinMaxScaler()

In [57]:
train_ = scaler.transform(train.values.reshape(-1,1))
test_ = scaler.transform(test.values.reshape(-1,1))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, root_mean_squared_error


def forecast_metrics(pred, truth):
    mae = mean_absolute_error(truth, pred)
    rmse = root_mean_squared_error(truth, pred)
    mape = mean_absolute_percentage_error(truth, pred)
    mse = mean_squared_error(truth, pred)

    print(f"  MSE:  {mse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAPE: {mape*100:.4f}")


In [59]:
train_ = pd.Series(train_.squeeze())
test_ = pd.Series(test_.squeeze())

In [60]:
arima = ArimaForecaster(train_data=train_, test_data=test_)
arima.fit()

2025-04-08 19:52:16,146 - INFO - ARIMA model training complete.
2025-04-08 19:52:16,146 - INFO - Best order: (1, 1, 0)


In [61]:
res = arima.forecast(horizon=1)

2025-04-08 19:52:16,151 - INFO - Performing ARIMA Forecast with horizon=1...
100%|██████████| 1969/1969 [12:19<00:00,  2.66it/s]
2025-04-08 20:04:35,566 - INFO - ARIMA forecast generated for 1969 total steps.


In [62]:
forecast_metrics(res, test_)

  MSE:  0.0001
  MAE:  0.0061
  RMSE: 0.0089
  MAPE: 1.2433
  MSE:  0.0001


# No Scale

In [63]:
arima = ArimaForecaster(train_data=train, test_data=test)
arima.fit()
res = arima.forecast(horizon=1)
forecast_metrics(res, test)

2025-04-08 20:04:55,722 - INFO - ARIMA model training complete.
2025-04-08 20:04:55,722 - INFO - Best order: (1, 1, 0)
2025-04-08 20:04:55,723 - INFO - Performing ARIMA Forecast with horizon=1...
100%|██████████| 1969/1969 [05:00<00:00,  6.55it/s]
2025-04-08 20:09:56,450 - INFO - ARIMA forecast generated for 1969 total steps.


  MSE:  17.0809
  MAE:  2.8110
  RMSE: 4.1329
  MAPE: 1.2368
  MSE:  17.0809


# Split, scale on train, apply transf on test

In [64]:
scaler = MinMaxScaler(feature_range=(0,1))

scaler.fit(train.values.reshape(-1,1))

MinMaxScaler()

In [65]:
train_ = scaler.transform(train.values.reshape(-1,1))
test_ = scaler.transform(test.values.reshape(-1,1))

In [68]:
print(train_.min(), train_.max())
print(test_.min(), test_.max())

0.0 0.9999999999999999
0.9399662574926198 7.009280387840434


In [69]:
train_ = pd.Series(train_.squeeze())
test_ = pd.Series(test_.squeeze())

In [70]:
arima = ArimaForecaster(train_data=train_, test_data=test_)
arima.fit()
res = arima.forecast(horizon=1)


2025-04-08 21:53:40,595 - INFO - ARIMA model training complete.
2025-04-08 21:53:40,596 - INFO - Best order: (1, 1, 0)
2025-04-08 21:53:40,597 - INFO - Performing ARIMA Forecast with horizon=1...
100%|██████████| 1969/1969 [09:04<00:00,  3.61it/s]
2025-04-08 22:02:45,310 - INFO - ARIMA forecast generated for 1969 total steps.


In [71]:
forecast_metrics(res, test_)


  MSE:  0.0039
  MAE:  0.0425
  RMSE: 0.0625
  MAPE: 1.2424
  MSE:  0.0039


In [81]:
# rmse test

test = np.ones(100) 
truth = np.random.random(100) 

In [82]:
root_mean_squared_error(truth, test)

0.5523399359352087

# Times test

In [1]:
import timesfm

ModuleNotFoundError: No module named 'timesfm'